### RAG Pipelines - Data Ingestion to vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\yellu\AppData\Local\Temp\ipykernel_22656\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
## Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")


    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFS in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 4 PDF files to process

Processing: attention.pdf
 Loaded 8 pages

Processing: embeddings.pdf
 Loaded 8 pages

Processing: object_detection.pdf
 Loaded 8 pages

Processing: proposal.pdf
 Loaded 7 pages

Total documents loaded: 31


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Educational document created for RAG learning\nPage 1\n Attention and Transformer Fundamentals\n A practical educational document for learning Retrieval-Augmented Generation (RAG)\nPurpose: This document introduces the attention mechanism and the Transformer architecture in a way that is useful\nfor understanding modern language models and RAG systems.\nLearning goals\n \x7f Understand why attention was introduced and what problem it solves.\n \x7f Understand queries, keys, and values (Q, K, V).\n \x7f Understand scaled dot-product attent

In [4]:
## text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 31 documents into 54 chunks

Example chunk:
Content: Educational document created for RAG learning
Page 1
 Attention and Transformer Fundamentals
 A practical educational document for learning Retrieval-Augmented Generation (RAG)
Purpose: This document ...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Educational document created for RAG learning\nPage 1\n Attention and Transformer Fundamentals\n A practical educational document for learning Retrieval-Augmented Generation (RAG)\nPurpose: This document introduces the attention mechanism and the Transformer architecture in a way that is useful\nfor understanding modern language models and RAG systems.\nLearning goals\n \x7f Understand why attention was introduced and what problem it solves.\n \x7f Understand queries, keys, and values (Q, K, V).\n \x7f Understand scaled dot-product attent

### Embedding And VectorstoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text string to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager




Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3644.21it/s]


Model loaded sucessfully. Embedding dimension: 384
